# Train OPT-125M for your existing PC Build Advisor
This notebook trains a genuine LoRA adapter for multilingual PC requirement extraction. It embeds the exact `nlu_data.py` and `opt_nlu.py` from your GitHub-ready package. No CSV upload is needed: this extraction task uses the project's labelled synthetic requests, not raw hardware listings. Your deployed recommendation engine still uses its existing CSVs.

**Google Colab:** Runtime → Change runtime type → T4 GPU, then run all cells. The first output ZIP contains the trained adapter at the exact repository path. The second ZIP adds genuine evaluation results. No GitHub token is needed. This notebook does not train OPT-350M, OPT-1.3B or an explanation adapter. Keep the full original notebook for those comparative experiments.

In [ ]:
import sys, subprocess, torch
from pathlib import Path
assert torch.cuda.is_available(), "Enable a T4 GPU in Colab before running this notebook."
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
# Match the deployed application's model libraries; retain Colab's CUDA-enabled torch.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0"])
print("Dependencies installed. If Colab requests a restart, restart the session and run all cells again.")


## 1. Restore the project's original NLP source files
These are the same prompts, JSON keys, dataset generator and LoRA routines that your project uses. They are written into the notebook working directory, not into your GitHub repository.

In [ ]:
from pathlib import Path
Path('nlu_data.py').write_text('"""\nnlu_data.py\n===========\nNatural-language understanding side of the PC Build Advisor.\n\n* SCHEMA / canonical JSON format that OPT must produce\n* Synthetic English / Malay / Manglish ("rojak") request generator\n  - phrases starting with \'*\' are HELD OUT: they never appear in the\n    training split and appear in ~50% of the test split, so the test\n    set measures generalisation, not memorisation.\n* Rule-based (regex / keyword) baseline extractor\n* Prompt builders and evaluation metrics\n"""\nimport json\nimport re\nimport random\n\nKEYS = ["use_case", "budget", "currency", "cpu_brand", "gpu_brand",\n        "ram_gb", "storage_gb", "extras", "compact"]\n\n# ---------------------------------------------------------------- phrase banks\nUSE = {\n    "gaming": [\n        ("gaming", "en"), ("playing AAA games", "en"), ("esports", "en"),\n        ("playing Valorant and CS2", "en"), ("*playing Cyberpunk and Elden Ring", "en"),\n        ("main game", "ms"), ("game berat", "ms"), ("game online", "ms"),\n        ("*main game FPS", "ms"), ("gaming lah", "mix"), ("play games one", "mix"),\n        ("*game Dota", "mix")],\n    "video_editing": [\n        ("video editing", "en"), ("editing videos", "en"), ("3D rendering", "en"),\n        ("content creation", "en"), ("*Premiere Pro and After Effects", "en"),\n        ("edit video", "ms"), ("edit video YouTube", "ms"), ("*render video", "ms"),\n        ("edit video lah", "mix"), ("do content creation", "mix"),\n        ("*Davinci Resolve", "mix")],\n    "office": [\n        ("office work", "en"), ("Excel and Word", "en"), ("school assignments", "en"),\n        ("*spreadsheets and emails", "en"), ("kerja pejabat", "ms"),\n        ("buat assignment", "ms"), ("kerja office", "ms"), ("*kerja Excel", "ms"),\n        ("office work only", "mix"), ("study one", "mix"), ("*do assignment", "mix")],\n    "programming": [\n        ("programming", "en"), ("coding", "en"), ("software development", "en"),\n        ("*web development and Docker", "en"), ("buat coding", "ms"),\n        ("develop aplikasi", "ms"), ("*program Python", "ms"),\n        ("coding lah", "mix"), ("do programming", "mix"), ("*code Java", "mix")],\n    "streaming": [\n        ("streaming", "en"), ("live streaming on Twitch", "en"),\n        ("game streaming", "en"), ("*streaming on YouTube", "en"),\n        ("buat live stream", "ms"), ("streaming game", "ms"),\n        ("*live stream TikTok", "ms"), ("stream game lah", "mix"),\n        ("*stream Twitch", "mix")],\n    "ai_ml": [\n        ("machine learning", "en"), ("AI model training", "en"),\n        ("deep learning", "en"), ("running local LLMs", "en"),\n        ("*data science with PyTorch", "en"), ("belajar machine learning", "ms"),\n        ("latih model AI", "ms"), ("*kerja deep learning", "ms"),\n        ("do AI stuff", "mix"), ("run LLM", "mix"), ("*train neural network", "mix")],\n    "general": [\n        ("daily use", "en"), ("general use", "en"),\n        ("browsing and watching Netflix", "en"), ("everyday tasks", "en"),\n        ("*light everyday work", "en"), ("guna harian", "ms"), ("guna biasa", "ms"),\n        ("tengok Netflix", "ms"), ("*guna rumah", "ms"), ("normal use only", "mix"),\n        ("surf internet", "mix"), ("*basic use", "mix")],\n}\nOPEN = {\n    "en": ["I want to build a PC for {u}.", "Can you recommend PC parts for {u}?",\n           "Need a computer for {u}.", "Help me pick components for a {u} PC.",\n           "*Looking to put together a desktop for {u}."],\n    "ms": ["Saya nak build PC untuk {u}.", "Tolong cadangkan komponen komputer untuk {u}.",\n           "Nak rakit PC untuk {u}.", "Boleh cadangkan PC untuk {u}?",\n           "*Saya perlukan komputer baru untuk {u}."],\n    "mix": ["Boss, I want build PC for {u} lah.", "Bro can recommend PC for {u} or not?",\n            "Eh got any PC build for {u} ah?", "Wah need new PC for {u} la.",\n            "*Hi ah, planning to build one PC for {u} leh."],\n}\nBUDGET = {\n    ("MYR", "en"): ["My budget is RM{n}", "Budget RM{n}", "around RM{n}",\n                    "under RM{n}", "*not more than RM{n}"],\n    ("MYR", "ms"): ["bajet RM{n}", "bawah RM{n}", "dalam RM{n}",\n                    "lebih kurang RM{n}", "*maksimum RM{n}", "bajet {n} ringgit"],\n    ("MYR", "mix"): ["budget RM{n} only", "RM{n} can ah", "{n} ringgit la",\n                     "budget {n} only", "*budget around RM{n} la"],\n    ("USD", "en"): ["my budget is ${n}", "budget ${n}", "under ${n}",\n                    "{n} USD", "*around {n} dollars"],\n}\nCPU = {"amd": ["AMD CPU", "Ryzen processor", "guna AMD", "*Ryzen only"],\n       "intel": ["Intel CPU", "Intel Core processor", "guna Intel", "*Intel chip please"]}\nGPU = {"nvidia": ["Nvidia GPU", "RTX card", "kad grafik Nvidia", "*NVIDIA graphics"],\n       "amd": ["Radeon GPU", "AMD graphics card", "kad grafik Radeon", "*AMD GPU"],\n       "intel": ["Intel Arc GPU", "Arc graphics"]}\nBOTH = {("amd", "amd"): ["all AMD build", "semua AMD", "full AMD", "*AMD everything"],\n        ("intel", "nvidia"): ["Intel and Nvidia combo", "Intel CPU with Nvidia card",\n                              "*Intel + RTX"]}\nRAM = ["{n}GB RAM", "RAM {n}GB", "{n} GB of memory", "ram {n}g", "*{n}gig memory"]\nSSD = {500: ["500GB SSD", "storage 500GB", "SSD 500GB"],\n       1000: ["1TB SSD", "storage 1TB", "SSD 1000GB", "*1 TB NVMe"],\n       2000: ["2TB SSD", "2TB storage", "SSD 2000GB", "*2 TB NVMe"]}\nEXTRA_WORDS = {"monitor": ["monitor", "monitor", "*display"],\n               "keyboard": ["keyboard", "keyboard", "papan kekunci"],\n               "mouse": ["mouse", "mouse", "tetikus"]}\nEXTRA_JOIN = {"en": ["with {x}", "include {x}", "also need {x}"],\n              "ms": ["dengan {x}", "nak {x} sekali"],\n              "mix": ["{x} sekali", "{x} also can", "got {x} or not"]}\nCOMPACT = ["small form factor", "compact build", "mini ITX", "PC kecil",\n           "small casing", "*as small as possible"]\n\n\ndef _pick(rng, items, mode, key=lambda x: x):\n    """Train mode never sees \'*\' items; test mode sees them ~50% of the time."""\n    star = [i for i in items if key(i).startswith("*")]\n    normal = [i for i in items if not key(i).startswith("*")]\n    if mode == "test" and star and rng.random() < 0.5:\n        choice = rng.choice(star)\n    else:\n        choice = rng.choice(normal)\n    return choice\n\n\ndef _clean(s):\n    return s[1:] if s.startswith("*") else s\n\n\n# ------------------------------------------------------------------ generator\ndef make_sample(rng, mode="train"):\n    """Return (text, label_dict)."""\n    lang = rng.choice(["en", "ms", "mix"], ) if False else rng.choice(["en", "ms", "mix"])\n    uc = rng.choice(list(USE))\n    uphrase, ulang = _pick(rng, USE[uc], mode, key=lambda t: t[0])\n    uphrase = _clean(uphrase)\n    opener = _clean(_pick(rng, OPEN[lang], mode)).format(u=uphrase)\n\n    cur = "USD" if rng.random() < 0.15 else "MYR"\n    if cur == "USD":\n        n = rng.choice(range(500, 3501, 100))\n        bl = "en"\n    else:\n        n = rng.choice(list(range(1500, 15001, 100)))\n        bl = lang\n    shown = str(n)\n    r = rng.random()\n    if n % 1000 == 0 and r < 0.35:\n        shown = f"{n // 1000}k"\n    elif n >= 1000 and r < 0.55:\n        shown = f"{n:,}"\n    elif cur == "MYR" and n % 500 == 0 and n >= 2000 and r < 0.75:\n        shown = f"{n / 1000:g}k"\n    if cur == "MYR" and bl != "en" and n % 1000 == 0 and rng.random() < 0.25:\n        bphrase = rng.choice(["bajet {k} ribu", "dalam {k} ribu", "lebih kurang RM{k} ribu"]\n                             ).format(k=n // 1000)\n    else:\n        bphrase = _clean(_pick(rng, BUDGET[(cur, bl)], mode)).format(n=shown)\n\n    label = dict(use_case=uc, budget=n, currency=cur, cpu_brand="any",\n                 gpu_brand="any", ram_gb=None, storage_gb=None, extras=[],\n                 compact=False)\n    clauses = [bphrase]\n\n    if rng.random() < 0.40:\n        t = rng.random()\n        if t < 0.25:\n            pair = rng.choice(list(BOTH))\n            label["cpu_brand"], label["gpu_brand"] = pair\n            clauses.append(_clean(_pick(rng, BOTH[pair], mode)))\n        else:\n            if rng.random() < 0.5:\n                b = rng.choice(list(CPU))\n                label["cpu_brand"] = b\n                clauses.append(_clean(_pick(rng, CPU[b], mode)))\n            if rng.random() < 0.7 or label["cpu_brand"] == "any":\n                b = rng.choice(list(GPU))\n                label["gpu_brand"] = b\n                clauses.append(_clean(_pick(rng, GPU[b], mode)))\n    if rng.random() < 0.35:\n        g = rng.choice([8, 16, 32, 64])\n        label["ram_gb"] = g\n        clauses.append(_clean(_pick(rng, RAM, mode)).format(n=g))\n    if rng.random() < 0.35:\n        g = rng.choice([500, 1000, 2000])\n        label["storage_gb"] = g\n        clauses.append(_clean(_pick(rng, SSD[g], mode)))\n    if rng.random() < 0.22:\n        ex = [e for e in EXTRA_WORDS if rng.random() < 0.5] or [rng.choice(list(EXTRA_WORDS))]\n        words = [_clean(_pick(rng, EXTRA_WORDS[e], mode)) for e in ex]\n        label["extras"] = [e for e in EXTRA_WORDS if e in ex]\n        x = ", ".join(words[:-1]) + (" and " if len(words) > 1 else "") + words[-1]\n        clauses.append(_clean(rng.choice(EXTRA_JOIN[lang])).format(x=x))\n    if rng.random() < 0.08:\n        label["compact"] = True\n        clauses.append(_clean(_pick(rng, COMPACT, mode)))\n\n    rng.shuffle(clauses)\n    tail = ""\n    for i, c in enumerate(clauses):\n        tail += ("" if i == 0 else rng.choice([", ", ". ", " and ", "; "])) + c\n    text = opener + " " + tail.strip() + rng.choice([".", "", "!", " thanks."])\n    if rng.random() < 0.12:\n        text = text.lower()\n    return text, label\n\n\ndef build_dataset(n, seed=0, mode="train"):\n    rng = random.Random(seed)\n    out, seen = [], set()\n    while len(out) < n:\n        t, l = make_sample(rng, mode)\n        if t in seen:\n            continue\n        seen.add(t)\n        out.append((t, l))\n    return out\n\n\n# ------------------------------------------------------- JSON / prompt format\ndef to_json(label):\n    """Canonical compact JSON with fixed key order."""\n    return json.dumps({k: label[k] for k in KEYS}, separators=(",", ":"))\n\n\ndef normalise(d):\n    """Make a parsed dict comparable to the gold label (fill + coerce)."""\n    out = dict(use_case="general", budget=None, currency="MYR", cpu_brand="any",\n               gpu_brand="any", ram_gb=None, storage_gb=None, extras=[],\n               compact=False)\n    if not isinstance(d, dict):\n        return out\n    for k in KEYS:\n        if k in d:\n            out[k] = d[k]\n    try:\n        out["budget"] = int(float(out["budget"])) if out["budget"] is not None else None\n    except (TypeError, ValueError, OverflowError):\n        out["budget"] = None\n    for k in ("ram_gb", "storage_gb"):\n        try:\n            out[k] = int(float(out[k])) if out[k] is not None else None\n        except (TypeError, ValueError, OverflowError):\n            out[k] = None\n    if not isinstance(out["extras"], list):\n        out["extras"] = []\n    out["extras"] = sorted(set(str(e) for e in out["extras"]) & {"monitor", "keyboard", "mouse"})\n    for key, choices, default in [\n        ("use_case", set(USE), "general"), ("currency", {"MYR", "USD"}, "MYR"),\n        ("cpu_brand", {"any", "intel", "amd"}, "any"),\n        ("gpu_brand", {"any", "nvidia", "amd", "intel"}, "any")]:\n        val = out[key]\n        out[key] = val if isinstance(val, str) and val in choices else default\n    for key in ("budget", "ram_gb", "storage_gb"):\n        if out[key] is not None and out[key] <= 0:\n            out[key] = None\n    out["compact"] = out["compact"] is True or str(out["compact"]).lower() == "true"\n    return out\n\n\ndef parse_model_output(text):\n    """Extract the first JSON object from generated text. None if invalid."""\n    text = text.strip()\n    start = text.find("{")\n    if start < 0:\n        return None\n    try:\n        obj, _ = json.JSONDecoder().raw_decode(text[start:])\n        return obj if isinstance(obj, dict) else None\n    except json.JSONDecodeError:\n        return None\n\n\nEXTRACT_HEAD = ("Extract the PC requirements from the customer message as JSON.\\n"\n                "Keys: use_case, budget, currency, cpu_brand, gpu_brand, ram_gb, "\n                "storage_gb, extras, compact.\\n")\n\n\ndef extract_prompt(text):\n    return f"{EXTRACT_HEAD}Message: {text}\\nJSON:"\n\n\ndef few_shot_prompt(text, shots):\n    body = "".join(f"Message: {t}\\nJSON: {to_json(l)}\\n\\n" for t, l in shots)\n    return f"{EXTRACT_HEAD}\\n{body}Message: {text}\\nJSON:"\n\n\ndef explain_prompt(facts):\n    return f"Recommended PC build:\\n{facts}\\nExplain why this build suits the customer:"\n\n\n# ----------------------------------------------------------- rule baseline\n_UC_RULES = [\n    ("ai_ml", r"machine learning|deep learning|\\bai\\b|\\bllm|neural|pytorch|data science|latih model"),\n    ("streaming", r"stream|twitch|live ?stream"),\n    ("video_editing", r"edit(ing)? video|video edit|render|premiere|davinci|content creation|after effects"),\n    ("programming", r"coding|\\bcode\\b|programming|\\bprogram\\b|develop|docker"),\n    ("gaming", r"gaming|\\bgame|esports|valorant|cs2|dota|fps|cyberpunk"),\n    ("office", r"office|excel|word|assignment|spreadsheet|pejabat|study"),\n    ("general", r"daily|general|netflix|everyday|harian|biasa|browsing|surf|basic|rumah"),\n]\n\n\ndef rule_extract(text):\n    """Keyword / regex baseline (no learning)."""\n    t = text.lower()\n    out = dict(use_case="general", budget=None, currency="MYR", cpu_brand="any",\n               gpu_brand="any", ram_gb=None, storage_gb=None, extras=[], compact=False)\n    for uc, pat in _UC_RULES:\n        if re.search(pat, t):\n            out["use_case"] = uc\n            break\n    m = re.search(r"(?:rm|\\$)\\s?([\\d][\\d,\\.]*)\\s?(k|ribu)?(?![a-z])|([\\d][\\d,\\.]*)\\s?(k|ribu)?(?![a-z])\\s?(?:ringgit|usd|dollars|myr)", t)\n    if not m:\n        m = re.search(r"(?:budget|bajet)\\s?(?:is |around |only )?([\\d][\\d,\\.]*)\\s?(k|ribu)?(?![a-z])", t)\n    if m:\n        groups = [g for g in m.groups() if g]\n        num = groups[0].replace(",", "")\n        try:\n            val = float(num.rstrip(".,")) * (1000 if len(groups) > 1 and groups[-1] in ("k", "ribu") else 1)\n            out["budget"] = int(val)\n        except ValueError:\n            pass\n    if "$" in t or re.search(r"\\busd\\b|dollars", t):\n        out["currency"] = "USD"\n    if re.search(r"all amd|semua amd|full amd|amd everything", t):\n        out["cpu_brand"] = out["gpu_brand"] = "amd"\n    elif re.search(r"intel and nvidia|intel cpu with nvidia|intel \\+ rtx", t):\n        out["cpu_brand"], out["gpu_brand"] = "intel", "nvidia"\n    else:\n        if re.search(r"ryzen|amd cpu|guna amd", t):\n            out["cpu_brand"] = "amd"\n        elif re.search(r"intel (cpu|core|chip)|guna intel", t):\n            out["cpu_brand"] = "intel"\n        if re.search(r"nvidia|rtx|geforce", t):\n            out["gpu_brand"] = "nvidia"\n        elif re.search(r"radeon|amd graphics|amd gpu", t):\n            out["gpu_brand"] = "amd"\n        elif re.search(r"arc", t):\n            out["gpu_brand"] = "intel"\n    m = re.search(r"(\\d+)\\s?(?:gb|g)\\s?(?:of )?(?:ram|memory)|ram\\s?(\\d+)\\s?gb?|(\\d+)\\s?gig", t)\n    if m:\n        out["ram_gb"] = int([g for g in m.groups() if g][0])\n    m = re.search(r"(\\d+(?:\\.\\d+)?)\\s?(tb|gb)\\s?(?:ssd|storage|nvme)|(?:ssd|storage)\\s?(\\d+(?:\\.\\d+)?)\\s?(tb|gb)", t)\n    if m:\n        g = [x for x in m.groups() if x]\n        v = float(g[0]) * (1000 if g[1] == "tb" else 1)\n        out["storage_gb"] = int(v)\n    for e, pat in (("monitor", r"monitor|display"), ("keyboard", r"keyboard|papan kekunci"),\n                   ("mouse", r"mouse|tetikus")):\n        if re.search(pat, t):\n            out["extras"].append(e)\n    out["compact"] = bool(re.search(r"small form|compact|mini itx|pc kecil|small casing|as small", t))\n    return out\n\n\n# ------------------------------------------------------------------ metrics\ndef evaluate(preds, golds):\n    """preds: list of dict|None (None = invalid JSON). golds: label dicts."""\n    n = len(golds)\n    if n == 0 or len(preds) != n:\n        raise ValueError("Predictions and gold labels must have equal nonzero length")\n    valid = sum(p is not None for p in preds)\n    field_ok = {k: 0 for k in KEYS}\n    exact = 0\n    for p, g in zip(preds, golds):\n        pn, gn = normalise(p), normalise(g)\n        ok_all = p is not None\n        for k in KEYS:\n            ok = (pn[k] == gn[k]) and p is not None\n            field_ok[k] += ok\n            ok_all &= ok\n        exact += ok_all\n    res = {"json_valid_%": 100 * valid / n, "exact_match_%": 100 * exact / n}\n    for k in KEYS:\n        res[f"{k}_%"] = 100 * field_ok[k] / n\n    res["mean_field_acc_%"] = sum(100 * field_ok[k] / n for k in KEYS) / len(KEYS)\n    return res\n\n\n# ------------------------------------------------------------------ OOD set\ndef _L(uc, budget, cur="MYR", cpu="any", gpu="any", ram=None, ssd=None,\n       extras=(), compact=False):\n    return dict(use_case=uc, budget=budget, currency=cur, cpu_brand=cpu,\n                gpu_brand=gpu, ram_gb=ram, storage_gb=ssd,\n                extras=sorted(extras), compact=compact)\n\n\n# Hand-written, natural-sounding requests that do NOT come from the generator\n# templates. Used to measure out-of-distribution robustness.\nOOD_SET = [\n    ("my son wants a pc to play fortnite and minecraft, we can spend about rm3500", _L("gaming", 3500)),\n    ("Looking for a workstation for Blender and Unreal Engine, 12k ringgit max, nvidia please", _L("video_editing", 12000, gpu="nvidia")),\n    ("recommend me a build for my office staff, excel and chrome only, rm 2000", _L("office", 2000)),\n    ("I\'d like to train small neural nets at home, budget 1800 usd, prefer nvidia", _L("ai_ml", 1800, "USD", gpu="nvidia")),\n    ("boss, rm5000 got any good gaming pc? intel cpu and 32gb ram", _L("gaming", 5000, cpu="intel", ram=32)),\n    ("pc untuk edit video tiktok, bajet 4 ribu", _L("video_editing", 4000)),\n    ("nak buat live stream dan main game, budget rm 6,500, nak monitor sekali", _L("streaming", 6500, extras=["monitor"])),\n    ("Can I get a decent PC for web dev around $1,200? AMD CPU please", _L("programming", 1200, "USD", cpu="amd")),\n    ("budget RM 3k, just for browsing youtube and zoom meetings", _L("general", 3000)),\n    ("Need something small, mini itx, for gaming. rm4500.", _L("gaming", 4500, compact=True)),\n    ("pls recommend pc 2tb ssd 64gb ram for data science, rm 14000", _L("ai_ml", 14000, ram=64, ssd=2000)),\n    ("I have 8k to spend. want to edit 4k videos with keyboard and mouse included.", _L("video_editing", 8000, extras=["keyboard", "mouse"])),\n    ("sis nak PC untuk kerja akaun, tak payah mahal, rm1800", _L("office", 1800)),\n    ("gaming pc rm 2.5k, amd gpu", _L("gaming", 2500, gpu="amd")),\n    ("Radeon only and ryzen, for playing Call of Duty, RM 6000", _L("gaming", 6000, cpu="amd", gpu="amd")),\n    ("Trying to spec a PC to run Llama locally, 3000 USD, 64GB of RAM", _L("ai_ml", 3000, "USD", ram=64)),\n    ("student pc for assignments and Zoom classes, RM 2200", _L("office", 2200)),\n    ("wah my budget only 1.5k ah, can build gaming pc or not", _L("gaming", 1500)),\n    ("twitch streamer here, need a build, ten thousand ringgit, 1tb ssd", _L("streaming", 10000, ssd=1000)),\n    ("Recommend a PC for coding and docker containers, 16GB RAM enough, RM4000", _L("programming", 4000, ram=16)),\n    ("build for editing youtube vids, budget about 5000 myr, nvidia gpu", _L("video_editing", 5000, gpu="nvidia")),\n    ("cheap pc for my mom to watch netflix and facebook, below rm 1800", _L("general", 1800)),\n    ("I want a high-end gaming rig, money is no object, about RM 20000", _L("gaming", 20000)),\n    ("Budget: 4,500 ringgit. Use: coding. Intel CPU + RTX 4060", _L("programming", 4500, cpu="intel", gpu="nvidia")),\n    ("pc game sekitar rm4k, ram 16gb, ssd 1tb", _L("gaming", 4000, ram=16, ssd=1000)),\n    ("need pc for rendering in 3ds max and v-ray, budget USD 2200", _L("video_editing", 2200, "USD")),\n    ("just a basic desktop for the shop counter, rm1600", _L("office", 1600)),\n    ("I stream on Twitch daily, 9000 MYR, with monitor keyboard and mouse", _L("streaming", 9000, extras=["monitor", "keyboard", "mouse"])),\n    ("Intel Arc GPU build for esports, 3000 ringgit", _L("gaming", 3000, gpu="intel")),\n    ("Fine-tune LLMs at home, RM 15000, Nvidia only", _L("ai_ml", 15000, gpu="nvidia")),\n    ("Can you suggest a quiet PC for general use? budget $900", _L("general", 900, "USD")),\n    ("mak nak PC untuk tengok drama dan video call, bajet rm2k", _L("general", 2000)),\n    ("program in C++ and run virtual machines, 32GB RAM, rm 5500", _L("programming", 5500, ram=32)),\n    ("1TB SSD for office use, rm2300", _L("office", 2300, ssd=1000)),\n    ("Gaming PC under 3k with Nvidia", _L("gaming", 3000, gpu="nvidia")),\n    ("I\'m into esports (Valorant), need a small PC, compact, budget RM 3500, ryzen", _L("gaming", 3500, cpu="amd", compact=True)),\n    ("Computer for gaming and I also need a monitor, RM 6500", _L("gaming", 6500, extras=["monitor"])),\n]\n', encoding="utf-8")
print('Restored nlu_data.py')


In [ ]:
from pathlib import Path
Path('opt_nlu.py').write_text('"""\nopt_nlu.py\n==========\nEverything that touches Meta\'s OPT models (Hugging Face port).\n\n  load_opt()            load a base OPT model in fp16\n  generate()            batched greedy generation\n  run_extraction()      zero-shot / few-shot requirement extraction\n  finetune_lora()       LoRA fine-tuning with prompt-masked loss\n  load_finetuned()      base + adapter -> merged model for inference\n  OptNLU                inference wrapper (extract + explain + fallbacks)\n  check_grounded()      lexical hallucination check for explanations\n"""\nimport gc\nimport json\nimport re\nimport time\n\nimport torch\n\nimport nlu_data as N\n\nOPT_SIZES = {"125m": "facebook/opt-125m", "350m": "facebook/opt-350m",\n             "1.3b": "facebook/opt-1.3b", "2.7b": "facebook/opt-2.7b"}\n\n\n# ------------------------------------------------------------------ loading\ndef load_opt(name, dtype=None):\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    dtype = dtype or (torch.float16 if torch.cuda.is_available() else torch.float32)\n    tok = AutoTokenizer.from_pretrained(name, use_fast=True)\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "left"\n    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)\n    model.to("cuda" if torch.cuda.is_available() else "cpu")\n    model.eval()\n    return tok, model\n\n\ndef free():\n    """Call AFTER `del model, tok` in the caller (del inside a function does\n    not drop the caller\'s references)."""\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef n_params(model):\n    return sum(p.numel() for p in model.parameters())\n\n\n# --------------------------------------------------------------- generation\n@torch.no_grad()\ndef generate(model, tok, prompts, max_new_tokens=100, batch_size=8, stop=None):\n    """Greedy decoding. Returns only the newly generated text per prompt."""\n    outs = []\n    tok.padding_side = "left"\n    for i in range(0, len(prompts), batch_size):\n        chunk = prompts[i:i + batch_size]\n        context = getattr(model.config, "max_position_embeddings", 2048)\n        tok.truncation_side = "left"\n        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,\n                  max_length=context - max_new_tokens).to(model.device)\n        kw = dict(max_new_tokens=max_new_tokens, do_sample=False,\n                  pad_token_id=tok.pad_token_id)\n        if stop:\n            kw.update(stop_strings=stop, tokenizer=tok)\n        try:\n            seq = model.generate(**enc, **kw)\n        except (TypeError, ValueError):       # old transformers: no stop_strings\n            kw.pop("stop_strings", None)\n            kw.pop("tokenizer", None)\n            seq = model.generate(**enc, **kw)\n        new = seq[:, enc["input_ids"].shape[1]:]\n        outs += [tok.decode(s, skip_special_tokens=True) for s in new]\n    return outs\n\n\ndef pick_shots(train, k=4, seed=0):\n    """k training examples with distinct use cases (for few-shot prompts)."""\n    import random\n    rng = random.Random(seed)\n    pool = train[:]\n    rng.shuffle(pool)\n    shots, seen = [], set()\n    for t, l in pool:\n        if l["use_case"] not in seen:\n            seen.add(l["use_case"])\n            shots.append((t, l))\n        if len(shots) == k:\n            break\n    return shots\n\n\ndef run_extraction(model, tok, texts, shots=None, batch_size=8):\n    """Returns (list of dict|None, seconds_per_example)."""\n    if shots:\n        prompts = [N.few_shot_prompt(t, shots) for t in texts]\n    else:\n        prompts = [N.extract_prompt(t) for t in texts]\n    t0 = time.time()\n    raw = generate(model, tok, prompts, max_new_tokens=128,\n                   batch_size=batch_size, stop=["}"])\n    dt = (time.time() - t0) / max(len(texts), 1)\n    preds = [N.parse_model_output(r) for r in raw]\n    return preds, dt, raw\n\n\n# ---------------------------------------------------------- fine-tuning\nclass _SFTData(torch.utils.data.Dataset):\n    def __init__(self, tok, pairs, max_len=640):\n        self.items = []\n        for prompt, completion in pairs:\n            p = tok(prompt, add_special_tokens=True)["input_ids"]\n            c = tok(" " + completion.strip(), add_special_tokens=False)["input_ids"]\n            c = c + [tok.eos_token_id]\n            if len(c) >= max_len:\n                raise ValueError("Completion exceeds max_len; increase the training context length")\n            p = p[-(max_len - len(c)):]\n            ids = p + c\n            labels = [-100] * len(p) + c    # loss only on completion\n            self.items.append((ids, labels))\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        return self.items[i]\n\n\ndef _collate(pad_id):\n    def fn(batch):\n        m = max(len(x[0]) for x in batch)\n        ids = torch.full((len(batch), m), pad_id, dtype=torch.long)\n        lab = torch.full((len(batch), m), -100, dtype=torch.long)\n        att = torch.zeros((len(batch), m), dtype=torch.long)\n        for i, (a, b) in enumerate(batch):          # right padding for training\n            ids[i, :len(a)] = torch.tensor(a)\n            lab[i, :len(b)] = torch.tensor(b)\n            att[i, :len(a)] = 1\n        return dict(input_ids=ids, labels=lab, attention_mask=att)\n    return fn\n\n\ndef finetune_lora(name, pairs, out_dir, epochs=2, lr=2e-4, batch_size=8,\n                  grad_accum=2, r=16, max_len=640, grad_ckpt=False, seed=42):\n    """LoRA fine-tune OPT `name` on (prompt, completion) pairs.\n    Returns (train_seconds, final_loss)."""\n    from transformers import (AutoTokenizer, AutoModelForCausalLM,\n                              TrainingArguments, Trainer, set_seed)\n    from peft import LoraConfig, get_peft_model\n    set_seed(seed)\n    if not pairs:\n        raise ValueError("No training pairs supplied")\n    cuda = torch.cuda.is_available()\n    tok = AutoTokenizer.from_pretrained(name, use_fast=True)\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16 if cuda else torch.float32)\n    model.to("cuda" if cuda else "cpu")\n    model.config.use_cache = False\n    if grad_ckpt:\n        model.gradient_checkpointing_enable()\n        model.enable_input_require_grads()\n    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none",\n                     target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],\n                     task_type="CAUSAL_LM")\n    model = get_peft_model(model, cfg)\n    for p in model.parameters():                      # fp16 base, fp32 adapters\n        if p.requires_grad:\n            p.data = p.data.float()\n    model.print_trainable_parameters()\n\n    ds = _SFTData(tok, pairs, max_len)\n    args = TrainingArguments(\n        output_dir=out_dir + "_ckpt", per_device_train_batch_size=batch_size,\n        gradient_accumulation_steps=grad_accum, num_train_epochs=epochs,\n        learning_rate=lr, lr_scheduler_type="cosine",\n        warmup_steps=max(1, int(0.05 * epochs * ((len(ds) + batch_size * grad_accum - 1) // (batch_size * grad_accum)))),\n        fp16=cuda, logging_steps=20, save_strategy="no", report_to="none",\n        remove_unused_columns=False, optim="adamw_torch", seed=seed)\n    trainer = Trainer(model=model, args=args, train_dataset=ds,\n                      data_collator=_collate(tok.pad_token_id))\n    t0 = time.time()\n    result = trainer.train()\n    secs = time.time() - t0\n    model.save_pretrained(out_dir)\n    tok.save_pretrained(out_dir)\n    loss = float(result.training_loss)\n    del model, trainer\n    free()\n    return secs, loss\n\n\ndef load_finetuned(name, adapter_dir):\n    """Base OPT + LoRA adapter, merged into fp16 weights for fast inference."""\n    from peft import PeftModel\n    tok, base = load_opt(name)\n    model = PeftModel.from_pretrained(base, adapter_dir)\n    model = model.merge_and_unload()\n    model.eval()\n    return tok, model\n\n\n# -------------------------------------------------------- training pairs\ndef extraction_pairs(dataset):\n    return [(N.extract_prompt(t), N.to_json(l)) for t, l in dataset]\n\n\ndef explanation_pairs(engine, dataset, pc_engine):\n    """(facts -> explanation) pairs from real engine output."""\n    pairs = []\n    for t, l in dataset:\n        res = engine.recommend(l)\n        if not res["ok"]:\n            continue\n        pairs.append((N.explain_prompt(pc_engine.facts_text(res)),\n                      pc_engine.template_explanation(res)))\n    return pairs\n\n\n# ----------------------------------------------------------- grounding check\n_NUM = re.compile(r"\\d[\\d,]*(?:\\.\\d+)?")\n_MODEL = re.compile(\n    r"\\b(?:RTX|GTX|RX|GT|Arc|Ryzen|Core(?: Ultra)?|Xeon|i[3579]-?|Radeon|GeForce)"\n    r"[\\w\\- ]*?\\d[\\w\\-]*", re.I)\n\n\ndef _sig_numbers(text):\n    out = set()\n    for m in _NUM.findall(text):\n        v = m.replace(",", "").rstrip(".")\n        if len(v.replace(".", "")) >= 2:        # ignore 1-digit numbers\n            out.add(v)\n    return out\n\n\ndef _models(text):\n    return {re.sub(r"\\s+", " ", m).strip().lower() for m in _MODEL.findall(text)}\n\n\ndef check_grounded(text, facts):\n    """Lexical grounding: every significant number and every CPU/GPU model\n    mentioned in `text` must also appear in `facts`.\n    Returns (is_grounded, unsupported_numbers, unsupported_models)."""\n    fnums = _sig_numbers(facts)\n    ftxt = facts.lower()\n    bad_n = sorted(n for n in _sig_numbers(text) if n not in fnums)\n    bad_m = sorted(m for m in _models(text) if m not in ftxt)\n    return (not bad_n and not bad_m), bad_n, bad_m\n\n\n# ----------------------------------------------------------- inference wrapper\nclass OptNLU:\n    """Extract requirements + explain builds with fine-tuned OPT, with\n    rule-based fallbacks so the demo never returns nothing."""\n\n    def __init__(self, extract_model, extract_tok, explain_model=None,\n                 explain_tok=None):\n        self.em, self.et = extract_model, extract_tok\n        self.xm, self.xt = explain_model or extract_model, explain_tok or extract_tok\n\n    def extract(self, text):\n        if self.em is None:\n            return N.rule_extract(text), True, "OPT not loaded"\n        preds, _, raw = run_extraction(self.em, self.et, [text], batch_size=1)\n        p = preds[0]\n        if p is None:\n            return N.rule_extract(text), True, raw[0]\n        p = N.normalise(p)\n        if p["budget"] is None:\n            fb = N.rule_extract(text)\n            p["budget"] = fb["budget"]\n            p["currency"] = fb["currency"]\n            return p, True, raw[0]\n        return p, False, raw[0]\n\n    def explain(self, res, pc_engine):\n        if self.xm is None:\n            return pc_engine.template_explanation(res), False, ([], [])\n        facts = pc_engine.facts_text(res)\n        txt = generate(self.xm, self.xt, [N.explain_prompt(facts)],\n                       max_new_tokens=260, batch_size=1)[0].split("\\n\\n")[0].strip()\n        ok, bn, bm = check_grounded(txt, facts)\n        if not ok or len(txt) < 40:\n            return pc_engine.template_explanation(res), False, (bn, bm)\n        return txt, True, ([], [])\n', encoding="utf-8")
print('Restored opt_nlu.py')


In [ ]:
import importlib, random, json, time, math, gc, csv, zipfile, shutil
import pandas as pd
import nlu_data as N
import opt_nlu as O
importlib.reload(N)
importlib.reload(O)
from pathlib import Path
MODEL = "facebook/opt-125m"
# Set to "quick" for a smoke test or keep "standard" for final single-model experiments.
MODE = "standard"
N_TRAIN, EPOCHS, N_TEST = (2000, 3, 100) if MODE == "standard" else (160, 1, 24)
assert MODE in {"standard", "quick"}
OUT = Path("opt125m_export")
ADAPTER = OUT / "adapters" / "extract_125m"
RESULTS = OUT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print(MODE, N_TRAIN, EPOCHS, N_TEST)


## 2. Prepare training and held-out examples
The project generator creates English, Malay and Manglish requests paired with structured JSON. Held-out patterns and a separate manually written set are used for evaluation. Missing-budget requests are not represented well in this generator; do not claim they are solved by training.

In [ ]:
train = N.build_dataset(N_TRAIN, seed=1, mode="train")
seen = {t.casefold().strip() for t, _ in train}
ood = list(N.OOD_SET)
ood_text = {t.casefold().strip() for t, _ in ood}
assert not (seen & ood_text), "Unexpected train/OOD overlap"
test = []
for t, label in N.build_dataset(N_TEST * 3, seed=2026, mode="test"):
    k = t.casefold().strip()
    if k not in seen and k not in ood_text:
        test.append((t, label))
        seen.add(k)
    if len(test) == N_TEST:
        break
assert len(test) == N_TEST
pairs = O.extraction_pairs(train)
print("Train / held-out synthetic / manually written:", len(train), len(test), len(ood))
print(pairs[0][0], pairs[0][1])


## 3. Fine-tune the actual Meta OPT-125M model
The pretrained model is downloaded from Hugging Face. LoRA updates a small set of trainable parameters; the original model is not uploaded to GitHub. This step requires a working internet connection and GPU.

In [ ]:
# Remove any stale adapter before training, so a failed run cannot be mistaken for a new one.
if ADAPTER.exists():
    shutil.rmtree(ADAPTER)
for name in ("OPT125M-adapter-GitHub.zip", "OPT125M-adapter-and-results.zip"):
    Path(name).unlink(missing_ok=True)
seconds, loss = O.finetune_lora(MODEL, pairs, str(ADAPTER), epochs=EPOCHS,
    lr=2e-4, batch_size=8, grad_accum=2, r=16, max_len=256, seed=42)
assert math.isfinite(loss), "Training loss is not finite. Do not deploy these weights."
pd.DataFrame([dict(model=MODEL, mode=MODE, train_samples=len(train), epochs=EPOCHS,
    lora_rank=16, final_training_loss=loss, training_seconds=seconds)]).to_csv(
    RESULTS / "opt125m_training.csv", index=False)
print("Training loss:", loss, "Training seconds:", seconds)


## 4. Verify and export the adapter
This reloads the trained model using the same loading routine as your deployed application. The adapter archive includes all saved tokenizer files. A finite training loss and a successful reload are not proof of extraction accuracy; use the next section for that.

In [ ]:
from safetensors import safe_open
assert (ADAPTER / "adapter_config.json").is_file()
weights = ADAPTER / "adapter_model.safetensors"
assert weights.is_file() and weights.stat().st_size > 0
cfg = json.loads((ADAPTER / "adapter_config.json").read_text())
assert cfg["base_model_name_or_path"] == MODEL
with safe_open(str(weights), framework="pt", device="cpu") as f:
    keys = list(f.keys())
    assert keys and all(torch.isfinite(f.get_tensor(k)).all().item() for k in keys)
    assert any("lora_B" in k and torch.count_nonzero(f.get_tensor(k)).item() > 0 for k in keys), "LoRA B weights are all zero"
O.free()
tok, model = O.load_finetuned(MODEL, str(ADAPTER))
example = "Nak PC untuk gaming, bajet RM4500, Ryzen CPU, 16GB RAM dan monitor."
preds, latency, raw = O.run_extraction(model, tok, [example], batch_size=1)
print("Smoke-test raw model output:", raw[0])
print("Parsed:", preds[0])
def export_zip(filename, include_results=False):
    with zipfile.ZipFile(filename, "w", compression=zipfile.ZIP_DEFLATED) as z:
        folders = [ADAPTER] + ([RESULTS] if include_results else [])
        for folder in folders:
            for p in sorted(folder.rglob("*")):
                if p.is_file():
                    z.write(p, p.relative_to(OUT).as_posix())
    print(filename, Path(filename).stat().st_size, "bytes")
export_zip("OPT125M-adapter-GitHub.zip")
print("Adapter ready to download in the Colab Files panel.")


## 5. Compare extraction performance
Evaluate the same two test sets for the rules, zero-shot pretrained model, few-shot pretrained model and fine-tuned adapter. Scores here are raw extraction results, without the deployment's budget correction/fallback. The manually written set is small; the synthetic set is generated, not real customer data. No scores are prefilled.

In [ ]:
metrics, predictions = [], []
sets = [("heldout_synthetic", test), ("manual_ood", ood)]
def record(method, name, examples, preds, sec, raw=None):
    row = dict(model=method, test_set=name, n=len(examples), seconds_per_example=sec)
    row.update(N.evaluate(preds, [g for _, g in examples]))
    metrics.append(row)
    for i, ((text, gold), pred) in enumerate(zip(examples, preds)):
        predictions.append(dict(method=method, test_set=name, text=text,
            gold=json.dumps(gold, ensure_ascii=False), prediction=json.dumps(pred, ensure_ascii=False),
            raw_output=raw[i] if raw is not None else ""))
for name, examples in sets:
    t0 = time.perf_counter()
    p = [N.rule_extract(t) for t, _ in examples]
    record("rule_baseline", name, examples, p, (time.perf_counter()-t0)/len(examples))
    p, sec, raw = O.run_extraction(model, tok, [t for t,_ in examples], batch_size=8)
    record("OPT125M_LoRA", name, examples, p, sec, raw)
del tok, model
O.free()
tok, model = O.load_opt(MODEL)
shots = O.pick_shots(train, k=4)
for name, examples in sets:
    for method, chosen in [("OPT125M_zero_shot", None), ("OPT125M_few_shot", shots)]:
        p, sec, raw = O.run_extraction(model, tok, [t for t,_ in examples], shots=chosen, batch_size=8)
        record(method, name, examples, p, sec, raw)
del tok, model
O.free()
frame = pd.DataFrame(metrics)
frame.to_csv(RESULTS / "opt125m_evaluation.csv", index=False)
pd.DataFrame(predictions).to_csv(RESULTS / "opt125m_predictions.csv", index=False)
display(frame[["model", "test_set", "json_valid_%", "exact_match_%", "mean_field_acc_%", "seconds_per_example"]])
(RESULTS / "opt125m_run_info.json").write_text(json.dumps(dict(model=MODEL, mode=MODE,
    train_count=len(train), heldout_count=len(test), manual_ood_count=len(ood), seed=42,
    note="Single-model requirement extraction; no explanation model trained."), indent=2))
export_zip("OPT125M-adapter-and-results.zip", include_results=True)


## 6. Download and deploy
Run the cell below to download the ZIP. If evaluation was interrupted, the first adapter-only ZIP is still available in Colab's Files panel after section 4 succeeds. Extract the ZIP on your computer; upload its `adapters/extract_125m/` folder into the existing repository root (merge with the existing `adapters/` folder). Upload the new `results/opt125m_*` files separately, without replacing your original baseline experiments. Commit changes. Keep Python 3.11 on Streamlit; choose **Fine-tuned OPT-125M (LoRA)** and inspect raw model output and extraction method. If it reports a rule-based fallback, the OPT path has not succeeded. The schema supports keyboard inclusion, not keyboard colour; this training does not add colour support.

In [ ]:
from google.colab import files
archive = Path("OPT125M-adapter-and-results.zip")
if not archive.exists():
    archive = Path("OPT125M-adapter-GitHub.zip")
assert archive.exists(), "Finish the training and export cells first."
files.download(str(archive))
